In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("../data/raw/all-data.csv", encoding="latin1", header=None, names=['sentiment', 'text'])
data['sentiment'].value_counts(normalize=True)

sentiment
neutral     0.594098
positive    0.281263
negative    0.124639
Name: proportion, dtype: float64

In [3]:
data.shape

(4846, 2)

In [4]:
data.head(790)

,sentiment,text
0,neutral,"According to Gran , the company has no plans t..."
1,neutral,Technopolis plans to develop in stages an area...
2,negative,The international electronic industry company ...
3,positive,With the new production plant the company woul...
4,positive,According to the company 's updated strategy f...
...,...,...
785,positive,The company estimates that the trend in demand...
786,positive,The estimated synergy benefits are at least EU...
787,neutral,The goal will be achieved via organic growth p...
788,positive,The Group 's business is balanced by its broad...


In [5]:
# I think it's better to have a removal_log to record all the misisng rows, duplicate rows, rows with no meaning, etc
columns = ['original_index', 'sentiment','text','removal_reason', 'decision']
removal_log = pd.DataFrame(columns=columns)

In [6]:
removal_log

,original_index,sentiment,text,removal_reason,decision


## Duplicate analysis:
*The dataset contained 6 exact duplicate rows where both the headline and sentiment label were identical. 
These were removed to avoid train/test leakage and inflated evaluation.
Additionally, 2 headlines appeared with conflicting sentiment labels. Since these examples introduce annotation noise and cannot be resolved objectively
without external context, all conflicting-label duplicate headlines were carefully examined and re-labelled for v1.
This keeps the evaluation cleaner and more defensible.*

In [7]:
# Let's create a helper function which maintains a removal log
def update_removal_log(df, indices, reason, decision, current_log):
    # 1. Let's check if the rows do exist 
    valid_indices = [idx for idx in indices if idx in df.index]
    if not valid_indices:
        print(f"Warning: These indices are not found and will be skipped {valid_indices}")
        return df, current_log # Return unchanged dataframes immediately

    # 2. Extract the rows based on the indices
    extracted_rows = df.loc[valid_indices].copy()
    extracted_rows['original_index'] = extracted_rows.index
    extracted_rows['removal_reason'] = reason
    extracted_rows['decision'] = decision
    
    # 3. Rearrange the columns order
    extracted_rows = extracted_rows[
        [
            'original_index',
            'sentiment',
            'text',
            'removal_reason',
            'decision'
        ]
    ]
    
    # 4. Append to log and drop from the original (WITHOUT inplace=True as this returns none)
    updated_removal_log = pd.concat([current_log, extracted_rows], ignore_index=True)
    updated_df = df.drop(index=valid_indices)
    return updated_df, updated_removal_log
    

In [8]:
# Let's first identify the rows
same_label_duplicates = data[data.duplicated(keep=False)]
removal_indices = [1099,1416,2396,2567,3094,3206]
cleaned_data, removal_log = update_removal_log(data, removal_indices, 'duplicate rows' , 'removed', current_log=removal_log)


*Now, let's solve the problem of the label-conflicts*
*There are 2 conflicts:*
 **Conflict 1: Neutral vs. Positive labels** 
 *The headline describes a company selling its stake to another company. Since the sentence does not state whether the sale was profitable, strategic, or harmful, I treated it as a **neutral** corporate transaction.*
 **Conflict 2: Positive vs. Neutral labels**
 *The headline describes the group as having a balanced business, broad portfolio, and presence in major markets. These are favorable business-positioning signals, so I retained the **positive** label.*
 

In [9]:
# Let's identify those rows
different_label_duplicates = data[data.duplicated(subset = ['text'], keep=False)]
removal_indices = [79,789]
#Let's drop each duplicate row
cleaned_data, removal_log = update_removal_log(cleaned_data, removal_indices, 'conflicting labels' , 'removed', current_log=removal_log)


**The class distribution did not change meaningfully after duplicate cleaning, so the cleaned dataset preserves the original class imbalance.**

In [10]:
cleaned_data['sentiment'].value_counts(normalize=True)

sentiment
neutral     0.593634
positive    0.281521
negative    0.124845
Name: proportion, dtype: float64

In [11]:
# Now let's check for the words with small length and also look for context_less or boilerplate text
filtered_df = cleaned_data[cleaned_data["text"].str.len() <=38]
filtered_df
# 2518 2554 2720 4368 4615 4228 4166

,sentiment,text
176,positive,Cargo volume grew by 7 % .
279,positive,Sales of clothing developed best .
329,positive,EPS grew to 0.04 eur from 0.02 eur .
388,positive,Turnover rose to EUR21m from EUR17m .
1023,neutral,All are welcome .
...,...,...
4228,neutral,Rapala Fishing Frenzy 2009 .
4368,neutral,"zip , '' experts warned Tuesday ."
4444,negative,EPS dropped to EUR0 .2 from EUR0 .3 .
4615,neutral,Liora 's got a brand-new bag .


In [12]:
meaningless_indices = [
    1023,
    1116,
    1461,
    1559,
    2356,
    2399,
    2471,
    2554,
    2569,
    2720,
    2983,
    3040,
    3061,
    3099,
    3425,
    3450,
    3464,
    3486,
    3720,
    3980,
    4228,
    4368,
    4615,
    4805
]
# These are all 25 rows   
cleaned_data, removal_log = update_removal_log(cleaned_data, meaningless_indices, 'contextless_or_boilerplate' , 'removed', current_log=removal_log)


In [13]:
cleaned_data['sentiment'].value_counts(normalize=True)

sentiment
neutral     0.591608
positive    0.282925
negative    0.125467
Name: proportion, dtype: float64

In [14]:
cleaned_data.shape

(4814, 2)

In [16]:
cleaned_data[cleaned_data.duplicated(subset=["text", "sentiment"], keep=False)].sum()

sentiment    
text         
dtype: str

In [21]:
conflict_check = (cleaned_data.groupby('text')['sentiment'].nunique())
(conflict_check > 1).sum()

np.int64(0)

In [22]:
majority_baseline = cleaned_data["sentiment"].value_counts(normalize=True).max()
majority_baseline

np.float64(0.5916078105525551)

In [24]:
cleaned_data.to_csv('../data/processed/cleaned_data.csv',index=False)
removal_log.to_csv('../reports/removal_log.csv', index=False)
print("Done, saved successfully")

Done, saved successfully
